# Combined 3B — gold-contract diagnostic
Uses the same pinned Qwen2.5-3B-Instruct FP16 generator and combined repair loop as the completed selective-facts and tool runs. Supplies each task's gold contract to BOTH generation and verification. No intent-parser model, adapter, training or old result is required. Parser calls/tokens are zero. Reference programs remain evaluator-only.

Attach **stage2_combined_gold_source.zip**, enable **Internet**, select **T4 x2**, then **Save Version → Run All**. Run two smoke tasks, 20 fresh development tasks and 100 fresh diagnostic test tasks, each in separate output folders. Smoke acceptance is not a quality threshold. Download **stage2_combined_gold_3b.zip**, the summary ZIP and the executed notebook. Partial bundles are not completed results.

Optional: attach the completed `stage2_combined_3b_selective_proposed.zip` and `stage2_combined_3b_tools_proposed.zip` as SEPARATE datasets for a strict paired comparison in the final cell. Automatically extracted datasets are supported. They are not used during inference. Environment/settings mismatches stop comparison, while the completed gold bundle remains available. You can instead send back the gold bundle for local analysis.

This measures the combined effect of gold contracts on generation and checking/repair. Initial code can differ from predicted-contract runs. Gold is an oracle diagnostic, not a deployable system or a guaranteed upper bound; previously inspected tasks are not fresh held-out evidence. Total cost excludes parser inference in this run; generation/repair costs are separately reported.


In [1]:
import datetime,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR=WORK/('run_'+RUN_ID);RUN_DIR.mkdir()
DRIVE_RESULTS=None
def save_results():
    if DRIVE_RESULTS is not None:
        DRIVE_RESULTS.mkdir(parents=True,exist_ok=True)
        for p in RUN_DIR.iterdir():
            if p.is_file() and p.suffix in ['.zip','.log','.json','.txt']:shutil.copy2(p,DRIVE_RESULTS/p.name)
        print('Drive results:',DRIVE_RESULTS)
print('Run folder:',RUN_DIR)



Run folder: /kaggle/working/run_20261115_085705_343867


In [2]:
MANIFEST_SHA='6f38ed25cd58df3e2c83e543a3b707ef1a68c46f0d401553b50176e225393be4'
ROOT=WORK/('stage2_combined_gold_source_'+MANIFEST_SHA[:12])
def unpack(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong source ZIP version'
    manifest=json.loads(raw);assert manifest['experiment']=='stage2-combined-gold-3b-fp16-v1'
    for name,sha in manifest['sha256'].items():
        relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name);assert hashlib.sha256(data).hexdigest()==sha,f'Changed source/input: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==data,f'Changed working source: {name}'
        else:path.write_bytes(data)
    sidecar=ROOT/'_stage2_combined_gold_manifest.json'
    if sidecar.exists():assert sidecar.read_bytes()==raw,'Changed source manifest'
    else:sidecar.write_bytes(raw)
    print('Verified fresh-run source/data:',len(manifest['sha256']),'files',MANIFEST_SHA)
found=False
for path in INPUT.rglob('manifest.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        unpack(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('stage2_combined_gold_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist()))
            prefix='stage2_combined_gold_source/';name=prefix+'manifest.json'
            if name not in z.namelist():continue
            raw=z.read(name)
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            unpack(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach/place stage2_combined_gold_source.zip in the input folder.'
PACKAGES=WORK/'stage2_combined_gold_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


Verified fresh-run source/data: 282 files 6f38ed25cd58df3e2c83e543a3b707ef1a68c46f0d401553b50176e225393be4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.33.3 requires PyYAML>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
d

In [3]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.is_available(); assert all(torch.cuda.get_device_properties(i).total_memory/2**30>=12 for i in range(torch.cuda.device_count())); print(torch.__version__,[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','eval.stage2_combined_3b.preflight'],cwd=ROOT,env=ENV,check=True)


Removing optional torchao 0.10.0: incompatible with PEFT 0.21.2; this workflow does not use torchao.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
{"torchao": null, "action": "removed_incompatible", "previous_version": "0.10.0"}
{"probe": "cpu_lora_injection_and_forward", "trainable_parameters": 16, "status": "passed"}
2.11.0+cu130 ['Tesla T4', 'Tesla T4']
{"event": "attention_visible_devices", "devices": [0, 1], "cuda_visible_devices": null}
{"event": "attention_shape", "device": "cuda:0", "query_shape": [1, 4, 32, 128], "original_key_shape": [1, 2, 32, 128], "expanded_key_shape": [1, 4, 32, 128], "dtype": "torch.float16", "native_gqa_expanded": true, "stage": "prefill", "has_mask": false, "is_causal": true, "backend": "EFFICIENT_ATTENTION", "math_fallback_allowed": false}
{"event": "kernel_self_test_passed", "device": "cuda:0", "case": "causal_prefill", "max_absolute_error": 0.001953125, "rtol": 0.005, "atol": 0.005

CompletedProcess(args=['/usr/bin/python3', '-m', 'eval.stage2_combined_3b.preflight'], returncode=0)

In [4]:
DEV_COUNT=20
def execute(smoke,split='test'):
    name='stage2_combined_gold_3b'+('_smoke' if smoke else '_dev' if split=='dev' else '')
    out=RUN_DIR/name;assert not out.exists(),'Use a fresh run folder.'
    command=[sys.executable,'-m','eval.stage2_combined_gold.runner','--method','gold','--preset','gold_contract','--out',str(out),'--packages',str(PACKAGES),'--split',split]
    if smoke:command+=['--smoke']
    log=RUN_DIR/(name+'.log')
    try:
        with log.open('w') as f:
            process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in process.stdout:
                print(line,end='');f.write(line);f.flush()
            code=process.wait()
    finally:save_results()
    assert code==0,f'Run failed: inspect {log} and the partial ZIP. A partial job is not a result.'
    assert json.loads((out/'completion.json').read_text())['complete']
    assert len(list((out/'tasks').glob('*.json')))==(2 if smoke else DEV_COUNT if split=='dev' else 100)
    return out.with_suffix('.zip')
SMOKE=execute(True)
DEV=execute(False,split='dev')
print('Development summary:',(DEV.with_suffix('')/'summary.json').read_text())
print('Smoke and end-to-end development completed. Inspect the development summary; the next cell runs 100 diagnostic test tasks freshly.')



Fetching 2 files: 100%|██████████| 2/2 [00:32<00:00, 16.45s/it]

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 123.66it/s]
GENERATE 1/2 adult_col_001
GENERATE 2/2 titanic_miss_004

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 124.75it/s]
DONE 1/2 adult_col_001: ACCEPT
DONE 2/2 titanic_miss_004: ACCEPT

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 123.62it/s]
GENERATE 1/20 adult_col_006
GENERATE 2/20 adult_conv_006
GENERATE 3/20 adult_dedup_006
GENERATE 4/20 adult_filt_006
GENERATE 5/20 adult_miss_006
GENERATE 6/20 airbnb_col_006
GENERATE 7/20 airbnb_conv_006
GENERATE 8/20 airbnb_dedup_006
GENERATE 9/20 airbnb_filt_006
GENERATE 10/20 airbnb_miss_006
GENERATE 11/20 taxi_col_006
GENERATE 12/20 taxi_conv_006
GENERATE 13/20 taxi_dedup_006
GENERATE 14/20 taxi_filt_006
GENERATE 15/20 taxi_miss_006
GENERATE 16/20 titanic_col_006
GENERATE 17/20 titanic_conv_006
GENERATE 18/20 titanic_dedup_006
GENERATE 19/20 titanic_filt_006
GENERATE 20/20 titanic_miss_006

Load

In [5]:
BUNDLE=execute(False)
OUT=RUN_DIR/'stage2_combined_gold_summary'
subprocess.run([sys.executable,'-m','eval.stage2_combined_gold.collect','--gold',str(BUNDLE),'--out',str(OUT)],cwd=ROOT,env=ENV,check=True)
from IPython.display import display,FileLink,Markdown
display(Markdown((OUT/'SUMMARY.md').read_text()))
display(FileLink(str(BUNDLE)));display(FileLink(str(OUT.with_suffix('.zip'))))
print('Download the gold bundle and this executed notebook. Predicted-contract outputs are optional in the next cell.')



Loading weights: 100%|██████████| 434/434 [00:03<00:00, 124.50it/s]
GENERATE 1/100 adult_col_001
GENERATE 2/100 adult_col_002
GENERATE 3/100 adult_col_003
GENERATE 4/100 adult_col_004
GENERATE 5/100 adult_col_005
GENERATE 6/100 adult_conv_001
GENERATE 7/100 adult_conv_002
GENERATE 8/100 adult_conv_003
GENERATE 9/100 adult_conv_004
GENERATE 10/100 adult_conv_005
GENERATE 11/100 adult_dedup_001
GENERATE 12/100 adult_dedup_002
GENERATE 13/100 adult_dedup_003
GENERATE 14/100 adult_dedup_004
GENERATE 15/100 adult_dedup_005
GENERATE 16/100 adult_filt_001
GENERATE 17/100 adult_filt_002
GENERATE 18/100 adult_filt_003
GENERATE 19/100 adult_filt_004
GENERATE 20/100 adult_filt_005
GENERATE 21/100 adult_miss_001
GENERATE 22/100 adult_miss_002
GENERATE 23/100 adult_miss_003
GENERATE 24/100 adult_miss_004
GENERATE 25/100 adult_miss_005
GENERATE 26/100 airbnb_col_001
GENERATE 27/100 airbnb_col_002
GENERATE 28/100 airbnb_col_003
GENERATE 29/100 airbnb_col_004
GENERATE 30/100 airbnb_col_005
GENERATE 3

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/stage2_combined_gold_source_6f38ed25cd58/eval/stage2_combined_gold/collect.py", line 136, in <module>
    a=p.parse_args();collect(a.gold,a.out,a.predicted)
                     ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/stage2_combined_gold_source_6f38ed25cd58/eval/stage2_combined_gold/collect.py", line 105, in collect
    gp,gr=read_bundle(gold,allow_scripted)
          ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/stage2_combined_gold_source_6f38ed25cd58/eval/stage2_combined_gold/collect.py", line 90, in read_bundle
    if c['extracted']['contract']!=inventory[row['task_id']][1]['gold_contract'] or c.get('source')!='task.gold_contract':raise ValueError('Gold contract differs from task')
                                                                                                               

CalledProcessError: Command '['/usr/bin/python3', '-m', 'eval.stage2_combined_gold.collect', '--gold', '/kaggle/working/run_20261115_085705_343867/stage2_combined_gold_3b.zip', '--out', '/kaggle/working/run_20261115_085705_343867/stage2_combined_gold_summary']' returned non-zero exit status 1.

In [ ]:
# Optional: attach the two COMPLETED combined result bundles to compare task by task.
# This cell skips comparison when neither result is attached. The gold run is independent.
paths=[]
for method in ['selective','tools']:
    name='stage2_combined_3b_'+method+'_proposed.zip'
    candidates=list(INPUT.rglob(name))
    # Support Kaggle auto-extracted datasets as well as intact ZIPs.
    for path in INPUT.rglob('plan.json'):
        try:
            plan=json.loads(path.read_text())
            if plan.get('experiment')!='stage2-combined-3b-fp16-v1' or plan.get('method')!=method or plan.get('smoke') or plan.get('split')!='test':continue
            folder=path.parent;seal=json.loads((folder/'bundle.json').read_text())
        except (OSError,ValueError):continue
        assert seal['complete'],'Incomplete predicted run'
        local=RUN_DIR/('predicted_'+method+'_'+str(len(candidates))+'.zip')
        with zipfile.ZipFile(local,'w',zipfile.ZIP_DEFLATED) as z:
            for name,digest in seal['sha256'].items():
                relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
                data=(folder/relative).read_bytes();assert hashlib.sha256(data).hexdigest()==digest
                z.writestr(name,data)
            z.writestr('bundle.json',json.dumps(seal))
        candidates.append(local)
    distinct={}
    for path in candidates:
        with zipfile.ZipFile(path) as z:seal=json.loads(z.read('bundle.json'))
        distinct[json.dumps(seal,sort_keys=True)]=path
    assert len(distinct)<=1,f'Multiple different full results for {method}'
    paths.extend(distinct.values())
if paths:
    OUT=RUN_DIR/'stage2_combined_gold_comparison'
    subprocess.run([sys.executable,'-m','eval.stage2_combined_gold.collect','--gold',str(BUNDLE),'--predicted',*map(str,paths),'--out',str(OUT)],cwd=ROOT,env=ENV,check=True)
    display(Markdown((OUT/'SUMMARY.md').read_text()));display(FileLink(str(OUT.with_suffix('.zip'))))
else:
    print('No predicted bundles attached. Gold results are complete; upload them for local comparison.')
